# Сравнение моделей с бейзлайном на CIC-IDS2017

In [36]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from cic_common import (
    LIGHT_TO_DARK_BLUE,
    MAIN_COLOR,
    RU,
    label_ru,
    load_dataset,
    show_table,
    style,
    with_spaces,
)

In [37]:
import os
os.chdir(
    '/Users/a.zhukov/dev/phd/notebooks'
)

## 1. Настройки

In [35]:
SAMPLE_SHARE = 1.0
MIN_FLOWS_PER_CLASS = 500
ATTEMPTED_IS_NORMAL = True
VAL_STARTS_AT = 0.6
TEST_STARTS_AT = 0.8
BASELINE_NAME = "Логистическая регрессия"
N_TREES = 200
RANDOM_STATE = 42

## 2. Данные и классы

Класс потока для модели: вид атаки или `BENIGN` для нормального трафика. Попытки атак по совету авторов датасета отнесены к норме (настройка `ATTEMPTED_IS_NORMAL`).

In [38]:
flows = load_dataset()

if not ATTEMPTED_IS_NORMAL:
    flows = flows[flows["Traffic Type"] != "Попытка атаки"]

# Класс потока: вид атаки, а для нормы и попыток атак BENIGN
flows["Class"] = np.where(flows["Traffic Type"] == "Атака", flows["Label"], "BENIGN")

# Момент, когда поток закончился. Длительность записана в микросекундах.
flows["Flow End"] = flows["Timestamp"] + pd.to_timedelta(flows["Flow Duration"], unit="us")


def take_sample(data, share, min_flows_per_class):
    """Оставляет заданную долю потоков каждого класса, но не меньше min_flows_per_class."""
    if share >= 1:
        return data
    kept = []
    for _, class_flows in data.groupby("Class"):
        count = max(round(len(class_flows) * share), min(len(class_flows), min_flows_per_class))
        kept.append(class_flows.sample(count, random_state=RANDOM_STATE))
    return pd.concat(kept)


flows = take_sample(flows, SAMPLE_SHARE, MIN_FLOWS_PER_CLASS)

# В файлах строки идут не строго по времени. Упорядочиваем: дальше время важно.
flows = flows.sort_values("Timestamp").reset_index(drop=True)

print("Потоков в расчёте:", with_spaces(len(flows)))
show_table(flows["Class"].value_counts().rename("Потоков").to_frame())

Потоков в расчёте: 2 099 976


,Потоков
Class,
Норма,1 594 545
Portscan,159 066
DoS Hulk,158 468
DDoS,95 144
Infiltration - Portscan,71 767
DoS GoldenEye,7 567
FTP-Patator,3 972
DoS Slowloris,3 859
SSH-Patator,2 961


## 3. Признаки

Пока только числовые описания поведения потока.

In [ ]:
NOT_FEATURES = {
    "id": "номер строки в файле",
    "Flow ID": "составлен из адресов и портов",
    "Src IP": "адрес стенда: большинство атак идёт с одного адреса",
    "Dst IP": "адрес стенда: большинство атак идёт на один адрес",
    "Src Port": "случайный порт клиента, поведение потока не описывает",
    "Dst Port": "на этом стенде порт выдаёт вид атаки",
    "Timestamp": "каждая атака записана в свои часы",
    "Flow End": "время, служебная колонка",
    "Day": "каждая атака записана в свой день",
    "Label": "ответ",
    "Attack": "ответ",
    "Traffic Type": "ответ",
    "Class": "ответ",
    "Attempted Category": "заполнена по метке, то есть содержит ответ",
    "Average Packet Size": "повторяет Packet Length Mean",
    "Fwd Segment Size Avg": "повторяет Fwd Packet Length Mean",
    "Bwd Segment Size Avg": "повторяет Bwd Packet Length Mean",
    "Packet Length Variance": "квадрат Packet Length Std",
}

feature_columns = [column for column in flows.columns if column not in NOT_FEATURES]

features = flows[feature_columns].copy()

# Два исправления по итогам EDA. Плохие значения заменяем пропусками,
# а пропуски потом заполняет сама модель по обучающей части.
# 1. Бесконечная скорость у потоков с нулевой длительностью.
features = features.replace([np.inf, -np.inf], np.nan)
# 2. Отрицательная длина заголовков.
for column in ["Fwd Header Length", "Bwd Header Length"]:
    features[column] = features[column].where(features[column] >= 0)

# float32 занимает вдвое меньше памяти, чем float64. Моделям такой точности хватает.
X = features.astype("float32")
del features

print("Признаков:", len(feature_columns))
show_table(pd.Series(NOT_FEATURES, name="Почему убрана").rename_axis("Колонка").reset_index())

## Разбиение как в WTMC2021

Здесь повторена схема из [кода авторов исправленной версии датасета](https://github.com/GintsEngelen/WTMC2021-Code), файлы `MakeDataNumpyFriendly.py` и `Benchmarking_RF.py`:

- потоки `Attempted` считаются `BENIGN`;
- адреса, порты, идентификатор потока и время в признаки не входят;
- строки с NaN и бесконечностями удаляются;
- разбиение случайное, со стратификацией по классу: 75% train, 25% test, `random_state=44`;
- случайный лес: 50 деревьев, глубина не больше 20;
- метрики: weighted precision, weighted recall, weighted F1.

Разбиение случайное

In [52]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

WTMC_TEST_SIZE = 0.25
WTMC_RANDOM_STATE = 44

# Берём значения признаков как есть, без исправлений из раздела 3:
# авторы ничего не исправляют, а только удаляют строки, где есть NaN или бесконечность.
wtmc_X = flows[feature_columns].astype("float32")
is_finite_row = np.isfinite(wtmc_X).all(axis=1).to_numpy()
wtmc_X = wtmc_X[is_finite_row]

# Модели работают с номерами классов. Номер класса это его место в списке wtmc_class_names.
wtmc_y, wtmc_class_names = pd.factorize(flows.loc[is_finite_row, "Class"], sort=True)

wtmc_X_train, wtmc_X_test, wtmc_y_train, wtmc_y_test = train_test_split(
    wtmc_X,
    wtmc_y,
    test_size=WTMC_TEST_SIZE,
    stratify=wtmc_y,
    random_state=WTMC_RANDOM_STATE,
)

print("Удалено строк с NaN или бесконечностью:", (~is_finite_row).sum())
print("Train:", with_spaces(len(wtmc_X_train)), "Test:", with_spaces(len(wtmc_X_test)))

# Сколько потоков каждого класса в train и в test
class_counts = pd.DataFrame(
    {"Train": np.bincount(wtmc_y_train), "Test": np.bincount(wtmc_y_test)},
    index=wtmc_class_names,
)
show_table(class_counts.sort_values("Train", ascending=False))

Удалено строк с NaN или бесконечностью: 5
Train: 1 574 978 Test: 524 993


,Train,Test
Норма,1 195 905,398 635
Portscan,119 299,39 767
DoS Hulk,118 851,39 617
DDoS,71 358,23 786
Infiltration - Portscan,53 825,17 942
DoS GoldenEye,5 675,1 892
FTP-Patator,2 979,993
DoS Slowloris,2 894,965
SSH-Patator,2 221,740
DoS Slowhttptest,1 305,435


In [53]:
def signed_log(values):
    """Сжимает разброс значений: log(1 + |x|) с сохранением знака.

    Такая же функция ещё раз объявлена в разделе 5.
    """
    return np.sign(values) * np.log1p(np.abs(values))


wtmc_models = {
    "DummyClassifier": DummyClassifier(strategy="most_frequent"),
    # Логистической регрессии и байесовскому классификатору важен масштаб признаков:
    # сжимаем разброс логарифмом и приводим признаки к общему масштабу.
    "Логистическая регрессия": make_pipeline(
        FunctionTransformer(signed_log), StandardScaler(), LogisticRegression(max_iter=300)
    ),
    "Наивный байесовский классификатор": make_pipeline(
        FunctionTransformer(signed_log), StandardScaler(), GaussianNB()
    ),
    # Параметры случайного леса как у авторов
    "Случайный лес (WTMC2021)": RandomForestClassifier(
        n_estimators=50, max_depth=20, n_jobs=-1, random_state=WTMC_RANDOM_STATE
    ),
    # У LightGBM по умолчанию нет ограничения на величину поправки в листе дерева.
    # На классах из нескольких потоков обучение из-за этого расходится, поэтому ограничения заданы явно.
    "LightGBM": LGBMClassifier(
        n_estimators=N_TREES,
        reg_lambda=1.0,
        min_child_weight=1.0,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=-1,
    ),
    "CatBoost": CatBoostClassifier(
        iterations=N_TREES, random_seed=RANDOM_STATE, verbose=0, allow_writing_files=False
    ),
}

wtmc_predictions, wtmc_fit_seconds = {}, {}
for name, model in wtmc_models.items():
    started = time.perf_counter()
    model.fit(wtmc_X_train, wtmc_y_train)
    wtmc_fit_seconds[name] = time.perf_counter() - started
    # ravel нужен для CatBoost: он возвращает предсказания столбцом, а не строкой
    wtmc_predictions[name] = np.asarray(model.predict(wtmc_X_test)).ravel()
    print(f"{name}: fit {wtmc_fit_seconds[name]:.1f} с")

DummyClassifier: fit 0.1 с
Логистическая регрессия: fit 156.0 с
Наивный байесовский классификатор: fit 2.5 с
Случайный лес (WTMC2021): fit 24.0 с
LightGBM: fit 36.8 с
CatBoost: fit 354.1 с


In [54]:
# Feature importance CatBoost: 10 самых важных признаков.
# Тип важности по умолчанию PredictionValuesChange: насколько в среднем меняется предсказание,
# когда меняется значение признака. Значения по всем признакам в сумме дают 100.
catboost_model = wtmc_models["CatBoost"]
importance = pd.Series(catboost_model.get_feature_importance(), index=catboost_model.feature_names_)

# Сортируем по возрастанию: на горизонтальном графике первая строка рисуется внизу,
# поэтому самый важный признак окажется наверху.
top_features = importance.nlargest(10).sort_values()

fig = go.Figure(
    go.Bar(
        x=top_features.values,
        y=[RU[column] for column in top_features.index],
        orientation="h",
        marker_color=MAIN_COLOR,
        hovertemplate="%{y}<br>%{x:.2f}<extra></extra>",
    )
)
style(
    fig,
    title="Feature importance CatBoost: 10 самых важных признаков",
    x_title="Feature importance (PredictionValuesChange)",
    height=460,
)
fig.show()

### Метрики на test

**Многоклассовые метрики.** Классы: `BENIGN` и каждый вид атаки отдельно.
- *Weighted Precision, Weighted Recall, Weighted F1*: метрика считается для каждого класса, потом берётся среднее с весами по числу потоков класса. Так считают авторы. Почти весь вес приходится на `BENIGN`.
- *Macro F1*: среднее F1 по классам без весов. Редкая атака весит столько же, сколько `BENIGN`.
- *Accuracy*: доля верно классифицированных потоков. Всегда равна Weighted Recall.

In [55]:
wtmc_rows = {}
for name, y_pred in wtmc_predictions.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        wtmc_y_test, y_pred, average="weighted", zero_division=0
    )
    wtmc_rows[name] = {
        "Weighted Precision": precision,
        "Weighted Recall": recall,
        "Weighted F1": f1,
        "Macro F1": f1_score(wtmc_y_test, y_pred, average="macro", zero_division=0),
        "Accuracy": accuracy_score(wtmc_y_test, y_pred),
        "Fit time, с": wtmc_fit_seconds[name],
    }

wtmc_scores = pd.DataFrame.from_dict(wtmc_rows, orient="index")
show_table(wtmc_scores, digits=4)

,Weighted Precision,Weighted Recall,Weighted F1,Macro F1,Accuracy,"Fit time, с"
DummyClassifier,"0,5766","0,7593","0,6554","0,0539","0,7593","0,0871"
Логистическая регрессия,"0,9910","0,9909","0,9907","0,8675","0,9909","155,9942"
Наивный байесовский классификатор,"0,9865","0,7368","0,8350","0,6956","0,7368","2,5287"
Случайный лес (WTMC2021),"0,9951","0,9951","0,9951","0,8957","0,9951","24,0371"
LightGBM,"0,9951","0,9951","0,9951","0,9488","0,9951","36,7731"
CatBoost,"0,9923","0,9924","0,9923","0,6941","0,9924","354,0567"


**Бинарные метрики.** Positive это любая атака, negative это `BENIGN`. Вид атаки не учитывается. FPR = FP / (FP + TN).

In [56]:
wtmc_benign_code = list(wtmc_class_names).index("BENIGN")
is_attack = wtmc_y_test != wtmc_benign_code

wtmc_binary_rows = {}
for name, y_pred in wtmc_predictions.items():
    predicted_attack = y_pred != wtmc_benign_code
    precision, recall, f1, _ = precision_recall_fscore_support(
        is_attack, predicted_attack, average="binary", zero_division=0
    )
    false_positives = (~is_attack & predicted_attack).sum()
    wtmc_binary_rows[name] = {
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": false_positives / (~is_attack).sum(),
    }

show_table(pd.DataFrame.from_dict(wtmc_binary_rows, orient="index"), digits=4)

,Precision,Recall,F1,FPR
DummyClassifier,"0,0000","0,0000","0,0000","0,0000"
Логистическая регрессия,"0,9991","0,9984","0,9987","0,0003"
Наивный байесовский классификатор,"0,4877","0,9990","0,6554","0,3326"
Случайный лес (WTMC2021),"1,0000","0,9997","0,9998","0,0000"
LightGBM,"0,9999","0,9999","0,9999","0,0000"
CatBoost,"0,9972","0,9948","0,9960","0,0009"


**F1 по классам.**

In [57]:
all_class_codes = np.arange(len(wtmc_class_names))

f1_by_class = pd.DataFrame(
    {
        name: f1_score(wtmc_y_test, y_pred, labels=all_class_codes, average=None, zero_division=0)
        for name, y_pred in wtmc_predictions.items()
    },
    index=wtmc_class_names,
)
f1_by_class.insert(0, "Support", np.bincount(wtmc_y_test, minlength=len(wtmc_class_names)))
show_table(f1_by_class.sort_values("Support", ascending=False), digits=4)

,Support,DummyClassifier,Логистическая регрессия,Наивный байесовский классификатор,Случайный лес (WTMC2021),LightGBM,CatBoost
Норма,398 635,"0,8632","0,9996","0,8004","0,9999","1,0000","0,9987"
Portscan,39 767,"0,0000","0,9458","0,9405","0,9684","0,9684","0,9633"
DoS Hulk,39 617,"0,0000","0,9995","0,9901","1,0000","1,0000","0,9986"
DDoS,23 786,"0,0000","0,9996","0,9993","1,0000","1,0000","0,9993"
Infiltration - Portscan,17 942,"0,0000","0,8635","0,8729","0,9283","0,9285","0,9129"
DoS GoldenEye,1 892,"0,0000","0,9892","0,6763","0,9984","0,9987","0,9238"
FTP-Patator,993,"0,0000","0,9975","0,9970","1,0000","1,0000","0,9831"
DoS Slowloris,965,"0,0000","0,9886","0,0230","0,9984","0,9984","0,9194"
SSH-Patator,740,"0,0000","0,9891","0,9870","0,9918","0,9980","0,9674"
DoS Slowhttptest,435,"0,0000","0,9828","0,0125","0,9920","0,9943","0,8487"
